In [1]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       691Mi       8.6Gi       2.0Mi       3.4Gi        11Gi
Swap:             0B          0B          0B


In [2]:
!pip install -q transformers datasets peft accelerate huggingface_hub gradio

In [3]:
#[Cell 1] Environment Setup & Authentication
from google.colab import userdata
from huggingface_hub import login
import warnings

# Suppress the specific warning about missing tokens
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

try:
    # Attempt to fetch the token from Colab's secure storage
    hf_token = userdata.get('HF_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.


In [4]:
#[Cell 2] Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load and configure the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load and configure the Neural Network Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       6.9Gi       251Mi        12Mi       5.6Gi       5.5Gi
Swap:             0B          0B          0B


In [6]:
# [Cell 2b] Baseline Inference (Before Fine-Tuning)
messages = [{"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}]

# Format the prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate an answer using the untrained base model
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract and print only the generated response
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("BASE MODEL ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASE MODEL ANSWER:
 I'm sorry, but I can't answer this question. This might be a sensitive and personal matter that should be discussed with a healthcare professional or contacted directly through their official channels. As an AI language model, I don't have access to private information


In [7]:
# Automatically download the dataset if it hasn't been uploaded manually
!wget -nc -q https://github.com/ML-Course-2026/session6/raw/refs/heads/main/material/datasets/MediCore.json

In [8]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       6.9Gi       238Mi        16Mi       5.6Gi       5.5Gi
Swap:             0B          0B          0B


In [9]:
#[Cell 3] Dataset Loading and Preprocessing
from datasets import load_dataset

raw_data = load_dataset("json", data_files="MediCore.json")

def preprocess(sample):
    messages = [
        {"role": "user", "content": sample['prompt']},
        {"role": "assistant", "content": sample['completion']}
    ]

    # Automatically applies <|im_start|> and <|im_end|> ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        #max_length=256,
        padding=False
    )
    # Explicitly create labels for loss calculation
    #tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(
    preprocess,
    remove_columns=raw_data["train"].column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

In [10]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       6.9Gi       245Mi        16Mi       5.5Gi       5.4Gi
Swap:             0B          0B          0B


In [11]:
# [Cell 4] LoRA Configuration
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

In [12]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       6.9Gi       248Mi        16Mi       5.5Gi       5.4Gi
Swap:             0B          0B          0B


In [13]:
#[Cell 5] Training Setup and Execution
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

# Let the collator handle padding + labels
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Split 10% of the data for validation
split = data["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    learning_rate=2e-4,

    per_device_train_batch_size=1,      # ↓ reduce to avoid OOM
    gradient_accumulation_steps=2,      # keeps effective batch size

    fp16=True,                          # ↓ big memory saver

    logging_steps=5,
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    remove_unused_columns=False
)

# IMPORTANT: enable memory savings
model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

Epoch,Training Loss,Validation Loss
1,0.626997,0.607730
2,0.416377,0.656084
3,0.214067,0.770979
4,0.175156,0.867896
5,0.112062,0.963945


('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

In [14]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       6.9Gi       248Mi        16Mi       5.5Gi       5.4Gi
Swap:             0B          0B          0B


In [15]:
# [Cell 6] Load Model for Testing
from peft import PeftModel, PeftConfig

path = "./my_qwen"
config = PeftConfig.from_pretrained(path)

# 1. Load the original, massive base model
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16
)

# 2. Attach your tiny, fine-tuned adapter to the base model
model = PeftModel.from_pretrained(base_model, path)

# 3. Re-enable caching for faster inference speeds
model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [16]:
# [Cell 7] Inference Execution
messages =[{"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}]

# Format the text with ChatML tags and generation prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert text to tensor numbers and move to GPU
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate the output
output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Strip out the input prompt so we only see the newly generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("FINE-TUNED ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

FINE-TUNED ANSWER:
 Dr. Samuel Kaarlo leads the neurology department at MediCore Hospital.


In [17]:
# [Cell 8] Merge and Save
# Fuse the adapter weights with the base model mathematically
merged_model = model.merge_and_unload()

# Save the unified, standalone model
merged_model.save_pretrained("./my_qwen_merged")
tokenizer.save_pretrained("./my_qwen_merged")
print("Model successfully merged and saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully merged and saved!


In [18]:
#  [Cell 9] [Modified Inference] Enforcing JSON Output
messages = [
    # 1. Add a system prompt with strict formatting rules
    {"role": "system", "content": "You are a helpful assistant. You must ONLY answer in valid JSON format. Do not include any plain text outside the JSON."},

    # 2. Add the user prompt
    {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}
]

# Apply the ChatML template (the tokenizer automatically handles the system role)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]
response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(response_text)

Dr. Samuel Kaarlo leads the neurology department at MediCore Hospital.


In [19]:
#  [Cell 10]
import gradio as gr

# 1. Define the function that Gradio will call when a user submits a prompt
def generate_response(user_prompt):
    messages = [
        # Improved System Prompt: Give the model an exact JSON structure to follow
        {
            "role": "system",
            "content": 'You are a helpful assistant. You must ONLY answer in valid JSON format using the following structure: {"answer": "your detailed response here"}'
        },
        {"role": "user", "content": user_prompt}
    ]

    # Format the text with ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text to tensor numbers and move to GPU
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate the output
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    # Strip out the input prompt
    generated_ids = output[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response_text

# 2. Build the Gradio Interface
demo = gr.Interface(
    fn=generate_response,                      # The function to run
    inputs=gr.Textbox(
        lines=3,
        placeholder="e.g. Who leads the neurology department at MediCore Hospital?",
        label="Enter your prompt here"
    ),
    outputs=gr.Textbox(label="Model Output"),  # Where the output will show
    title="MediCore Fine-Tuned Qwen Bot",
    description="Ask questions about MediCore hospital. The model is instructed to reply in JSON format."
)

# 3. Launch the app (share=True creates a public link you can open)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5f690bbe1149fc7bc1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5f690bbe1149fc7bc1.gradio.live


In [20]:
#  [Cell 11]
import gradio as gr
import json

def generate_response(user_prompt):
    # Removed the system prompt since the model wasn't trained to use one
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_ids = output[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # --- PYTHON JSON WRAPPER ---
    # We take the raw text and force it into a JSON dictionary
    json_output = json.dumps({"answer": response_text}, indent=4)

    return json_output

demo = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=3, label="Enter your prompt here"),
    outputs=gr.Code(language="json", label="JSON Output"), # Changed output to code block
    title="MediCore Fine-Tuned Qwen Bot"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://baf9cc3946526db09f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://baf9cc3946526db09f.gradio.live


In [21]:
#  [Cell 12]
import gradio as gr
from pydantic import BaseModel, Field

# 1. Define your strict Pydantic Schema
class HospitalResponse(BaseModel):
    # You can add as many fields as you want here
    answer: str = Field(description="The main text answer to the user's question")
    model_version: str = Field(default="Qwen2.5-1.5B-MediCore", description="The model used")

def generate_response(user_prompt):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate the text
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_ids = output[0][inputs.input_ids.shape[1]:]

    # 1. Get the RAW plain text from the model
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # 2. Pass the raw text into your Pydantic model
    structured_response = HospitalResponse(answer=raw_text)

    # 3. Use Pydantic to dump it into a perfect JSON string
    json_output = structured_response.model_dump_json(indent=4)

    return json_output

# Build the Gradio Interface
demo = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=3, label="Enter your prompt here"),
    outputs=gr.Code(language="json", label="Pydantic JSON Output"),
    title="MediCore Fine-Tuned Qwen Bot (Pydantic Powered)"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b9de0bc0806e5c57d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b9de0bc0806e5c57d4.gradio.live
